# 01 — Exploratory Data Analysis

First look at the raw HR dataset before any cleaning: what the distributions look like,
which features separate leavers from stayers, and what the class balance forces us to do
later when choosing a metric.

**Input:** `data/raw/hr_training_data.csv` (5,000 rows, generated by `scripts/generate_training_data.py`)

In [ ]:
# Run from the repo root or from ai-service/ — this resolves either way.
import sys, pathlib

root = pathlib.Path.cwd()
while root != root.parent and not (root / 'app' / 'etl').exists():
    root = root.parent
sys.path.insert(0, str(root))
print('ai-service root:', root)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 40)
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv(root / 'data' / 'raw' / 'hr_training_data.csv')
print(df.shape)
df.head()

## Schema and completeness

In [ ]:
df.info()

missing = df.isnull().sum()
print('\nMissing values per column:')
print(missing[missing > 0] if missing.any() else 'none')

## Class balance

This is the number that decides how we evaluate. With a minority class well under 50%,
plain accuracy is misleading — a model that predicts "nobody leaves" would score well
and be useless. AUC-ROC and recall are the metrics that matter for this problem.

In [ ]:
counts = df['attrition'].value_counts().sort_index()
rate = df['attrition'].mean()
print(counts)
print(f'\nAttrition rate: {rate:.2%}')
print(f'Majority-class baseline accuracy: {1 - rate:.2%}  <- beat this, or the model adds nothing')

counts.plot(kind='bar', color=['#52c41a', '#ff4d4f'], rot=0)
plt.xticks([0, 1], ['Stayed', 'Left'])
plt.title('Class balance')
plt.ylabel('Employees')
plt.show()

## Numeric distributions

In [ ]:
FEATURES = [
    'salary', 'tenureMonths', 'engagementScore', 'performanceScore',
    'absenteeismDays', 'overtimeHours', 'lastPromotionMonths', 'trainingHours',
]

df[FEATURES].describe().T

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 14))
for ax, col in zip(axes.ravel(), FEATURES):
    sns.histplot(df[col], bins=40, ax=ax, kde=True, color='#1677ff')
    ax.set_title(col)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

## Which features actually separate the classes?

Overlapping distributions mean a feature carries little signal on its own. Look for pairs
where the two curves sit apart — those are what the model will lean on.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 14))
for ax, col in zip(axes.ravel(), FEATURES):
    for label, name, colour in [(0, 'Stayed', '#52c41a'), (1, 'Left', '#ff4d4f')]:
        sns.kdeplot(df.loc[df['attrition'] == label, col], ax=ax,
                    label=name, fill=True, alpha=0.35, color=colour)
    ax.set_title(col)
    ax.set_xlabel('')
    ax.legend()
plt.tight_layout()
plt.show()

### Standardised mean difference

A single number per feature for the same question: how many pooled standard deviations
apart are the two groups? Bigger magnitude = more univariate signal.

In [ ]:
rows = []
for col in FEATURES:
    a = df.loc[df['attrition'] == 1, col]
    b = df.loc[df['attrition'] == 0, col]
    pooled = np.sqrt((a.var() + b.var()) / 2)
    rows.append({'feature': col, 'left_mean': a.mean(), 'stayed_mean': b.mean(),
                 'cohens_d': (a.mean() - b.mean()) / pooled if pooled else 0.0})

sep = pd.DataFrame(rows).assign(abs_d=lambda d: d['cohens_d'].abs())
sep = sep.sort_values('abs_d', ascending=False).drop(columns='abs_d')
sep

In [ ]:
order = sep.reindex(sep['cohens_d'].abs().sort_values().index)
colours = ['#ff4d4f' if v > 0 else '#1677ff' for v in order['cohens_d']]
plt.figure(figsize=(9, 5))
plt.barh(order['feature'], order['cohens_d'], color=colours)
plt.axvline(0, color='#555', lw=1)
plt.xlabel("Cohen's d  (positive = higher among leavers)")
plt.title('Univariate separation by feature')
plt.tight_layout()
plt.show()

## Correlation

Checked for two reasons: features correlated with the target are candidates, and features
heavily correlated with *each other* are redundant.

In [ ]:
corr = df[FEATURES + ['attrition']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

print('Correlation with attrition, strongest first:')
print(corr['attrition'].drop('attrition').sort_values(key=abs, ascending=False))

## Outliers

The ETL caps rather than drops (IQR × 1.5). Dropping would discard real high earners and
genuinely overworked employees — exactly the people the model needs to score.

In [ ]:
plt.figure(figsize=(13, 6))
scaled = (df[FEATURES] - df[FEATURES].mean()) / df[FEATURES].std()
sns.boxplot(data=scaled, orient='h', color='#1677ff')
plt.title('Standardised distributions (outliers visible)')
plt.xlabel('z-score')
plt.tight_layout()
plt.show()

for col in FEATURES:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f'{col:22s} {n:4d} outliers ({n / len(df):.1%})')

## Cross-column consistency

`lastPromotionMonths` must not exceed `tenureMonths` — you cannot have gone longer without
a promotion than you have been employed. `clean.check_consistency` repairs these.

In [ ]:
bad = df['lastPromotionMonths'] > df['tenureMonths']
print(f'Inconsistent rows: {bad.sum()} ({bad.mean():.2%})')
df.loc[bad, ['tenureMonths', 'lastPromotionMonths']].head()

---
## Takeaways

1. **Class imbalance is real** — evaluate on AUC-ROC and recall, not accuracy, and set
   `scale_pos_weight` when training.
2. **Engagement and overtime carry the most univariate signal**, which the trained model's
   gain ranking later confirms.
3. **No feature separates the classes cleanly on its own** — the interactions are where the
   remaining signal lives, which is why the ETL engineers 4 ratio features.
4. **Outliers are capped, not dropped** — they are legitimate employees.

Next: `02_etl_exploration.ipynb`.